In [1]:
%matplotlib widget
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
from specutils.utils.wcs_utils import air_to_vac
from astropy import units as u
from astropy import constants as const

In [21]:
#Units ['AA', 'erg/s/cm2/AA', 'erg/s/cm2/AA.1', '-', 'erg/s/cm2/AA.2',
#      'erg/s/cm2/AA.3', '-.1', 'erg/s/cm2/AA.4', '-.2', 'erg/s/cm2/AA.5',
#      'cm^-1']
P20251125 = pd.read_csv('Output_spectras/20251125_stack_A.csv', skiprows=1)
P20251126 = pd.read_csv('Output_spectras/20251126_stack_A.csv', skiprows=1)
P20251127 = pd.read_csv('Output_spectras/20251127_stack_A.csv', skiprows=1)
post = pd.read_csv('Output_spectras/Full_Stack_post_stack_A.csv', skiprows=0)

P20250922 = pd.read_csv('Output_spectras/20250922_stack_A.csv', skiprows=1)
P20250923 = pd.read_csv('Output_spectras/20250923_stack_A.csv', skiprows=1)
P20250924 = pd.read_csv('Output_spectras/20250924_stack_A.csv', skiprows=1)
pre = pd.read_csv('Output_spectras/Last_3nights_stack_A.csv', skiprows=1)

In [22]:
post.keys()

Index(['WAVE', 'FLUX_STACK', 'ERR_STACK', 'NCONTRIB', 'FLUX_STACK_SKYSUB',
       'ERR_STACK_SKYSUB', 'NCONTRIB_SKYSUB', 'CONTINUUM', 'REFLECTANCE',
       'KURUCZ_SHIFTED', 'WAVENUMBER'],
      dtype='object')

In [23]:
df = {'Wave': [], 'REL_Intensity': [], 'MOL': [], 'identifier': []
}
with open('Mol_lines/neowise_lines.txt', 'r') as f:
    for line in f.readlines():
        #split by any wthitespace
        line = line.strip().split(' ')
        #take put all the '' elements
        line = [x for x in line if x != '']
        # now lets keep the first three elements as they are and fuse all the rest as the identifier
        line = [float(line[0]), float(line[1]), line[2], ' '.join(line[3:])]
        #append to the table
        df['Wave'].append(line[0])
        df['REL_Intensity'].append(line[1])
        df['MOL'].append(line[2])
        df['identifier'].append(line[3])

neowise_lines = pd.DataFrame(df)
neowise_lines_CN = neowise_lines[neowise_lines['MOL']=='CN']
neowise_lines_CH = neowise_lines[neowise_lines['MOL']=='CH']
neowise_lines_NH2 = neowise_lines[neowise_lines['MOL']=='NH2']
neowise_lines_C_3_ = neowise_lines[neowise_lines['MOL']=='C_3_']
neowise_lines_C_2 = neowise_lines[neowise_lines['MOL']=='C_2_']

#selesct the strongest by ordering by REL_Intensity
neowise_lines_CN = neowise_lines_CN.sort_values(by='REL_Intensity', ascending=False).head(10)
neowise_lines_CH = neowise_lines_CH.sort_values(by='REL_Intensity', ascending=False).head(15)
neowise_lines_NH2 = neowise_lines_NH2.sort_values(by='REL_Intensity', ascending=False).head(100)
neowise_lines_C_3_ = neowise_lines_C_3_.sort_values(by='REL_Intensity', ascending=False).head(1000)
neowise_lines_C_2 = neowise_lines_C_2.sort_values(by='REL_Intensity', ascending=False).head(200)

#now update the initial neowise_lines
neowise_lines = pd.concat([neowise_lines_CN, neowise_lines_CH, neowise_lines_NH2, neowise_lines_C_3_, neowise_lines_C_2])

In [24]:
import numpy as np
import astropy.units as u
iron = pd.read_csv('Element_lines/iron_lines_air.txt', delim_whitespace=True, names=['WAVE', 'name', 'ion'])
nickel = pd.read_csv('Element_lines/nickel_lines_air.txt', delim_whitespace=True, names=['WAVE', 'name', 'ion'])
def plot_grouped_lines(ax, neowise_lines, molecule="CH",
                       tol=0.3,  # grouping tolerance in Å
                       y_text=1.1e-13,
                       color="g", linestyle="--", alpha=0.5,
                       fontsize=6, limx=3800, limx2=8000):
    """
    Plot spectral lines for a given molecule on the provided axis, but
    group lines that are closer than `tol` Å into a single representative line.

    Parameters
    ----------
    ax : matplotlib axis
        Axis where the lines will be drawn.
    neowise_lines : pandas DataFrame
        Must contain columns 'Wave', 'MOL', 'identifier'.
    molecule : str
        Molecule substring to filter lines (default 'CH').
    tol : float
        Maximum wavelength separation (Å) to group lines together.
    y_text : float
        Y-position of the label text (in data coordinates).
    color, linestyle, alpha, fontsize : styling parameters.

    Returns
    -------
    None
    """

    # 1) Filter for the selected molecule and convert to vacuum wavelength
    lines = []
    for _, row in neowise_lines.iterrows():
        if molecule not in row["MOL"]:
            continue
        if not (limx <= row["Wave"] <= limx2):
            continue
        λ_vac = air_to_vac(row["Wave"] * u.AA).to_value(u.AA)
        label = f"{row['MOL']}"
        lines.append((λ_vac, label))

    if len(lines) == 0:
        return  # nothing to plot

    # 2) Sort by wavelength
    lines.sort(key=lambda x: x[0])

    # 3) Group nearby lines
    groups = []
    current = [lines[0]]

    for λ, label in lines[1:]:
        if abs(λ - current[-1][0]) < tol:
            current.append((λ, label))
        else:
            groups.append(current)
            current = [(λ, label)]

    groups.append(current)  # last one

    # 4) Plot a representative for each group
    for group in groups:
        λ_center = np.mean([g[0] for g in group])

        # You can combine labels or choose the first
        # label = ", ".join([g[1] for g in group])
        label = group[0][1]

        # Draw line and label
        ax.axvline(λ_center, color=color, linestyle=linestyle, alpha=alpha)


/var/folders/qk/w_n1xnwj22176_k_hx_3sdf80000gn/T/ipykernel_95216/563384843.py:3: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  iron = pd.read_csv('Element_lines/iron_lines_air.txt', delim_whitespace=True, names=['WAVE', 'name', 'ion'])
/var/folders/qk/w_n1xnwj22176_k_hx_3sdf80000gn/T/ipykernel_95216/563384843.py:4: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  nickel = pd.read_csv('Element_lines/nickel_lines_air.txt', delim_whitespace=True, names=['WAVE', 'name', 'ion'])


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
iron = pd.read_csv('Element_lines/iron_lines_air.txt', delim_whitespace=True, names=['WAVE', 'name', 'ion'])
nickel = pd.read_csv('Element_lines/nickel_lines_air.txt', delim_whitespace=True, names=['WAVE', 'name', 'ion'])
# --- Helper: wavelength cut ---------------------------------------------------
def cut(data, liminf=5000, limsup=9000):
    mask = (data['WAVE'] > liminf) & (data['WAVE'] < limsup)
    return data['WAVE'][mask], data['FLUX_STACK'][mask], data['CONTINUUM'][mask]


plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14

# --- Your spectra DataFrames (already defined) --------------------------------
# P20251125, P20251126, P20251127,
# P20250924, P20250923, P20250922
# each with columns 'WAVE' and 'FLUX_STACK'

spectra = {
    "P20251125": P20251125,
    "P20251126": P20251126,
    "P20251127": P20251127,
    "P20250924": P20250924,
    "P20250923": P20250923,
    "P20250922": P20250922,
}

dates = {"P20251125": "2025-11-25",
         "P20251126": "2025-11-26",
         "P20251127": "2025-11-27",
         "P20250924": "2025-09-24",
         "P20250923": "2025-09-23",
         "P20250922": "2025-09-22",
}
# Order in which we’ll stack them (top to bottom)
spec_order_black = ['P20251127', 'P20251126', 'P20251125']
spec_order_red   = ['P20250924', 'P20250923', 'P20250922']
spec_order_all   = spec_order_black + spec_order_red

# --- Desired vertical positions as FRACTIONS of the axis height --------------
# 1.0 = top, 0.0 = bottom in data space
fraction_levels = {
    'P20251127': 0.8,
    'P20251126': 0.58,
    'P20251125': 0.32,
    'P20250924': 0.18,
    'P20250923': 0.10,
    'P20250922': 0.04,
}


def prepare_window_data(spectra_dict, wmin, wmax):
    """
    For a given wavelength window, cut and normalize all spectra.
    Returns:
        x_dict[name], y_norm_dict[name], (ymin_raw, ymax_raw)
    where ymin_raw/ymax_raw are the global min/max of all normalized spectra.
    """
    x_dict = {}
    y_norm_dict = {}
    all_vals = []

    for name in spec_order_all:
        x, y, c = cut(spectra_dict[name], wmin, wmax)
        y_norm = y - c  # remove local baseline
        x_dict[name] = x
        y_norm_dict[name] = y_norm
        all_vals.append(y_norm)

    ymin_raw = min(arr.min() for arr in all_vals)
    ymax_raw = max(arr.max() for arr in all_vals)

    return x_dict, y_norm_dict, (ymin_raw, ymax_raw)


def compute_ylim(ymin_raw, ymax_raw, margin_factor=1.2):
    """
    Choose a y-limits range for one window based on its raw min/max.
    margin_factor > 1 enlarges the range a bit for nicer spacing.
    """
    center = 0.5 * (ymin_raw + ymax_raw)
    half_span = 0.5 * (ymax_raw - ymin_raw) * margin_factor
    if half_span == 0:
        half_span = 1.0  # fallback if flat

    # your asymmetric choice
    return center - half_span * 0.5, center + half_span


def get_offset_for_fraction(y_norm, ylim, frac):
    """
    Given normalized spectrum y_norm, axis limits ylim=(ymin, ymax),
    and a target fraction frac in [0, 1], compute the offset to
    place the median of y_norm at that fraction of the axis.
    """
    ymin, ymax = ylim
    y_target = ymin + frac * (ymax - ymin)
    return y_target - np.median(y_norm)


def plot_window_with_fraction_offsets(ax, spectra_dict, wmin, wmax, date,margin_factor=6):
    """
    Plot one wavelength window on a given axis, with spectra positioned
    at fixed fractions of the axis height so that they align across panels.
    """
    # 1) Prepare normalized data and raw range
    x_dict, y_norm_dict, (ymin_raw, ymax_raw) = prepare_window_data(
        spectra_dict, wmin, wmax
    )

    # 2) Decide ylim for THIS window (panel-specific scaling)
    ylim = compute_ylim(ymin_raw, ymax_raw, margin_factor=margin_factor)
    ax.set_ylim(*ylim)

    # 3) Plot each spectrum at its desired axis fraction
    # Black group
    for name in spec_order_black:
        frac = fraction_levels[name]
        y_norm = y_norm_dict[name]
        offset = get_offset_for_fraction(y_norm, ylim, frac)
        y_plot = y_norm + offset
        ax.plot(x_dict[name], y_plot, color='crimson', alpha=0.7,
                label=name if wmin == 3800 else None, zorder= 100)
        if date:    
            ax.text(0.491*(wmin+wmax), ylim[0]+frac*(ylim[1]-ylim[0])-1e-13, dates[name], fontsize=13)
    # Red group
    for name in spec_order_red:
        frac = fraction_levels[name]
        y_norm = y_norm_dict[name]
        offset = get_offset_for_fraction(y_norm, ylim, frac)
        y_plot = y_norm + offset
        ax.plot(x_dict[name], y_plot, color='steelblue', alpha=0.7,
                label=name if wmin == 3800 else None, zorder=100)
        if date:
            ax.text(0.491*(wmin+wmax), ylim[0]+frac*(ylim[1]-ylim[0])-8e-14, dates[name], fontsize=13)
    ax.set_xlim(wmin, wmax)

def plot_three_windows_concat(ax, spectra_dict, windows, margin_factor=2.5, gap=5.0):
    """
    Plot multiple wavelength windows concatenated on a fake x-axis,
    keeping Pre/Post aligned vertically using fraction_levels.
    
    windows: list of (wmin, wmax)
    spectra_dict: {"Post": post, "Pre": pre}
    """

    # we'll use Pre/Post as defined later in your script
    local_spec_order = ['Post', 'Pre']

    # 1) Cut and normalize all windows, collect y for a global ylim
    data = {}        # data[win_idx][name] = (x, y_norm)
    all_y_norm = []

    for i, (wmin, wmax) in enumerate(windows):
        data[i] = {}
        for name in local_spec_order:
            x, y, c = cut(spectra_dict[name], wmin, wmax)
            y_norm = y - c
            data[i][name] = (x, y_norm)
            all_y_norm.append(y_norm)

    ymin_raw = min(arr.min() for arr in all_y_norm)
    ymax_raw = max(arr.max() for arr in all_y_norm)

    # 2) Decide a single ylim for this concatenated panel
    ylim = compute_ylim(ymin_raw, ymax_raw, margin_factor=margin_factor)
    ax.set_ylim(*ylim)

    # 3) Build fake x-axis with gaps between windows
    offsets = []
    x_offset = 0.0
    for (wmin, wmax) in windows:
        offsets.append(x_offset)
        width = wmax - wmin
        x_offset += width + gap

    total_length = x_offset - gap  # last window ends here

    # For x-ticks: centers of each window
    window_centers = [
        off + 0.5 * (wmax - wmin)
        for off, (wmin, wmax) in zip(offsets, windows)
    ]

    # 4) Plot each spectrum in each window at its fraction level
    color_map = {'Post': 'crimson', 'Pre': 'steelblue'}

    for name in local_spec_order:
        frac = fraction_levels[name]  # uses your global fraction_levels for Pre/Post
        color = color_map[name]
        for (wmin, wmax), off, i in zip(windows, offsets, range(len(windows))):
            x, y_norm = data[i][name]
            # map real wavelength → fake axis: start each window at its offset
            x_fake = (x - wmin) + off
            offset_y = get_offset_for_fraction(y_norm, ylim, frac)
            y_plot = y_norm + offset_y
            ax.plot(x_fake, y_plot, color=color, alpha=0.7)
            air = 5577.346680
            x = air_to_vac(air*u.AA)
            x = x.to(u.AA).value
            x = (x - wmin) + off
            ax.axvline(x, linestyle='-', color='k', alpha=0.2)
            ax.text(x-0.15, 1.25e-13, 'Comet [OI]', rotation=90, fontsize=13, alpha=0.2)
            air = 6300.308594
            x = air_to_vac(air*u.AA)
            x = x.to(u.AA).value
            x = (x - wmin) + off
            ax.axvline(x, linestyle='-', color='k', alpha=0.2)
            ax.text(x-0.15, 1.25e-13, 'Comet [OI]', rotation=90, fontsize=13, alpha=0.2)
            air = 6363.782715
            x = air_to_vac(air*u.AA)
            x = x.to(u.AA).value
            x = (x - wmin) + off
            ax.axvline(x, linestyle='-', color='k', alpha=0.2)
            ax.text(x-0.15, 1.25e-13, 'Comet [OI]', rotation=90, fontsize=13, alpha=0.2)

            vel = [np.mean([-21.669884, -21.327189, -20.957300]), np.mean([-4.365752, -4.172302, -3.999497])]
            k=0
            for v in vel:
                if k==0:
                    alt=1.25e-13
                    k+=1
                else:
                    alt=0.25e-13
                air = 5577.346680
                x = air_to_vac(air*u.AA)
                x = x.to(u.AA)
                x = (x * (1 - v * u.km/u.s / const.c)).to(u.AA).value
                x = (x - wmin) + off
                ax.axvline(x, linestyle='-', color='k', alpha=0.2)
                ax.text(x+0.05, alt, 'Sky [OI]', rotation=90, fontsize=13, alpha=0.2)
                air = 6300.308594
                x = air_to_vac(air*u.AA)
                x = x.to(u.AA)
                x = (x * (1 - v * u.km/u.s / const.c)).to(u.AA).value
                x = (x - wmin) + off
                ax.axvline(x, linestyle='-', color='k', alpha=0.2)
                ax.text(x+0.05, alt, 'Sky [OI]', rotation=90, fontsize=13, alpha=0.2)
                air = 6363.782715
                x = air_to_vac(air*u.AA)
                x = x.to(u.AA)
                x = (x * (1 - v * u.km/u.s / const.c)).to(u.AA).value
                x = (x - wmin) + off
                ax.axvline(x, linestyle='-', color='k', alpha=0.2)
                ax.text(x+0.05, alt, 'Sky [OI]', rotation=90, fontsize=13, alpha=0.2)

    ax.set_xlim(0, total_length)

    # 5) Label x-ticks with the actual window ranges or centers
    tick_labels = [f"{wmin:.1f}–{wmax:.1f}" for (wmin, wmax) in windows]
    ax.set_xticks(window_centers)
    ax.set_xticklabels(tick_labels, rotation=0, fontsize=13)

# --- Figure setup: 5 x 2 grid (leftmost column empty) ------------------------
fig = plt.figure(figsize=(30, 12))

gs = GridSpec(nrows=2, ncols=5, figure=fig, wspace=0.25, hspace=0.25)

# EMPTY axis spanning 2 rows in the first (left) column
ax_empty = fig.add_subplot(gs[:, 0])   # col 0, both rows

# Big axes spanning 2 rows for the next two columns
ax_big_0 = fig.add_subplot(gs[:, 1])  # col 1, both rows
ax_big_1 = fig.add_subplot(gs[:, 2])  # col 2, both rows

# 2 x 2 small axes in the last two columns
ax_small_00 = fig.add_subplot(gs[0, 3])  # row 0, col 3
ax_small_10 = fig.add_subplot(gs[1, 3])  # row 1, col 3
ax_small_01 = fig.add_subplot(gs[0, 4])  # row 0, col 4
ax_small_11 = fig.add_subplot(gs[1, 4])  # row 1, col 4


# --- Assign wavelength windows ------------------------------------
# First two columns (tall)
plot_window_with_fraction_offsets(ax_big_0, spectra, 4040, 4420, False)  # your first window
plot_window_with_fraction_offsets(ax_big_1, spectra, 5140, 5720, False)  # your second window
plot_window_with_fraction_offsets(ax_empty, spectra, 3800, 3990, False)  # your first window

minus = {
    'P20251127': 0.026,
    'P20251126': 0.026,
    'P20251125': 0.026,
    'P20250924': 0.0224,
    'P20250923': 0.0224,
    'P20250922': 0.016,
}

for key in dates.keys():
    date = dates[key]
    rel = fraction_levels[key]
    minu = minus[key]
    ax_empty.text(
        0.98, rel-minu,              # (x_rel, y_rel) in axes coordinates
        date,
        transform=ax_empty.transAxes,
        fontsize=13,
        ha='right', va='center'
    )

# --- Labels -------------------------------------------------------
ax_empty.set_ylabel("Flux + offsets (arbitrary units)", fontsize=16)
ax_small_00.set_ylabel("Flux + offsets (arbitrary units)", fontsize=14)
ax_small_10.set_ylabel("Flux + offsets (arbitrary units)", fontsize=14)
for ax in [ax_empty, ax_big_0, ax_big_1, ax_small_00, ax_small_10, ax_small_01, ax_small_11]:
    ax.set_xlabel("Wavelength [Å]", fontsize=16)

spectra = {
    "Post": post,
    "Pre": pre,
}

dates = {"Post": "Post Perihilion Stack",
         "Pre": "Pre Perihilion Stack",
}
# Order in which we’ll stack them (top to bottom)
spec_order_black = ['Post']
spec_order_red   = ['Pre']
spec_order_all   = spec_order_black + spec_order_red

# --- Desired vertical positions as FRACTIONS of the axis height --------------
# 1.0 = top, 0.0 = bottom in data space
fraction_levels = {
    'Pre': 0.15,
    'Post': 0.54,
}

plot_window_with_fraction_offsets(ax_small_00, spectra, 3867, 3883.5, False, 2.7) 
ax_small_00.text(
    0.23, 0.4,              # (x_rel, y_rel) in axes coordinates
    "Post Perihelion Stack",
    transform=ax_small_00.transAxes,
    fontsize=13,
    ha='center', va='center'
)

ax_small_00.text(
    0.23, 0.06,              # (x_rel, y_rel)
    "Pre Perihelion Stack",
    transform=ax_small_00.transAxes,
    fontsize=13,
    ha='center', va='center'
)
plot_window_with_fraction_offsets(ax_small_01, spectra, 4296, 4320, False, 2.6)

fraction_levels = {
    'Pre': 0.15,
    'Post': 0.59,
}
plot_window_with_fraction_offsets(ax_small_10, spectra, 5155, 5167.5, False, 2.3) 
windows_three = [
    (5578.5, 5579.5),
    (6301.75, 6302.75),
    (6365.2, 6366.2),
]
fraction_levels = {
    'Pre': 0.07,
    'Post': 0.48,
}
plot_three_windows_concat(ax_small_11, spectra, windows_three, margin_factor=2.6, gap=.1)

#lines

ax_small_00.text(
    0.3, 0.97,              # (x_rel, y_rel) in axes coordinates
    r"CN B$^2 \Sigma^+$-X$^2 \Sigma^+$ (Violet Band)",
    transform=ax_small_00.transAxes,
    fontsize=13,
    ha='center', va='center'
)

for idx, row in iron.iterrows():
    x = row['WAVE']
    x = air_to_vac(row['WAVE']* u.AA)
    x = x.to_value(u.AA)
    if x < 3867.5 or x > 3883:
        continue
    ax_small_00.axvline(x, color='k', linestyle=':', alpha=0.2)
    ax_small_00.text(x-0.55, 1.6e-13, f"{row['name']} {row['ion']}", rotation=90, fontsize=13, alpha=0.7)

ax_small_00.axvspan(3867.5, 3876, color='orange', alpha=0.05)
ax_small_00.text(
    3875, 1.4e-13,              # (x_rel, y_rel) in axes coordinates
    r"R-Branch",
    fontsize=13,
    ha='right', va='center'
)

ax_small_00.axvspan(3877, 3883, color='darkorchid', alpha=0.05)
ax_small_00.text(
    3877, 1.4e-13,              # (x_rel, y_rel) in axes coordinates
    r"P-Branch",
    fontsize=13,
    ha='left', va='center'
)

for idx, row in iron.iterrows():
    x = row['WAVE']
    x = air_to_vac(row['WAVE']* u.AA)
    x = x.to_value(u.AA)
    if x < 4296 or x > 4320:
        continue
    ax_small_01.axvline(x, color='k', linestyle=':', alpha=0.2)
    ax_small_01.text(x-1, 7.5e-14, f"{row['name']} {row['ion']}", rotation=90, fontsize=13, alpha=0.7)

for idx, row in neowise_lines.iterrows():
    if 'CH' not in row['MOL']:
        continue

    x = row['Wave']
    if x < 4297 or x > 4306:
        continue
    x = air_to_vac(row['Wave']* u.AA)
    x = x.to_value(u.AA)
    ax_small_01.axvline(x, color='k', linestyle='--', alpha=0.2)
    if x>4305:
        continue
    ax_small_01.text(x-1, 7.5e-14, f"{row['MOL']}", rotation=90, fontsize=13, alpha=0.7)

plot_grouped_lines(ax_small_01, neowise_lines, molecule="CH",
                       tol=0.3,
                       y_text=7.5e-14,
                       color="k", linestyle="--", alpha=0.2,
                       fontsize=13,
                       limx=4306, limx2=4320)

for idx, row in neowise_lines.iterrows():
    if 'CH' not in row['MOL']:
        continue
    x = row['Wave']
    if x < 4306 or x > 4320:
        continue
    x = air_to_vac(row['Wave']* u.AA)
    x = x.to_value(u.AA)
    ax_small_01.text(x-2.4, 7.5e-14, f"{row['MOL']}", rotation=90, fontsize=13, alpha=0.7)
    break

plot_grouped_lines(ax_small_10, neowise_lines, molecule="C_2",
                       tol=0.1,
                       y_text=7.5e-14,
                       color="k", linestyle="--", alpha=0.2,
                       fontsize=13,
                       limx=5154, limx2=5156)
plot_grouped_lines(ax_small_10, neowise_lines, molecule="C_2",
                       tol=0.1,
                       y_text=7.5e-14,
                       color="k", linestyle="--", alpha=0.2,
                       fontsize=13,
                       limx=5157, limx2=5160)
plot_grouped_lines(ax_small_10, neowise_lines, molecule="C_2",
                       tol=0.15,
                       y_text=7.5e-14,
                       color="k", linestyle="--", alpha=0.2,
                       fontsize=13,
                       limx=5160, limx2=5163)
# plot_grouped_lines(ax_small_10, neowise_lines, molecule="C_2",
#                        tol=0.15,
#                        y_text=7.5e-14,
#                        color="k", linestyle="--", alpha=0.2,
#                        fontsize=13,
#                        limx=5163, limx2=5166)
ax_small_10.axvspan(5164.5, 5166.7, color='gray', alpha=0.2)

ax_small_10.text(
    0.29, 0.975,              # (x_rel, y_rel) in axes coordinates
    r"C$_2$",
    transform=ax_small_10.transAxes,
    fontsize=13,
    ha='center', va='center'
)
ax_small_10.text(
    0.9, 0.975,              # (x_rel, y_rel) in axes coordinates
    r"C$_2$",
    transform=ax_small_10.transAxes,
    fontsize=13,
    ha='center', va='center'
)

for idx, row in iron.iterrows():
    x = row['WAVE']
    x = air_to_vac(row['WAVE']* u.AA)
    x = x.to_value(u.AA)
    if x < 4040 or x > 4420:
        continue
    ax_big_0.axvline(x, color='k', linestyle=':', alpha=0.2)

xs = []
for idx, row in neowise_lines.iterrows():
    if 'CH' not in row['MOL']:
        continue
    x = row['Wave']
    if x < 4297 or x > 4306:
        continue
    x = air_to_vac(row['Wave']* u.AA)
    x = x.to_value(u.AA)
    xs.append(x)

ax_big_0.axvspan(min(xs), max(xs), color='gray', alpha=0.1)
ax_big_0.axvspan(4313, 4315.7, color='gray', alpha=0.1)

ax_big_0.text(min(xs)-13, 5.5e-13, s=f"CH", rotation=90, fontsize=13, alpha=0.7)
ax_big_0.text(4054, 5.5e-13, s=f"Fe I", rotation=90, fontsize=13, alpha=0.7)
ax_big_0.text(4120, 5.5e-13, s=f"Fe I", rotation=90, fontsize=13, alpha=0.7)
ax_big_0.text(4260, 5.5e-13, s=f"Fe I", rotation=90, fontsize=13, alpha=0.7)
ax_big_0.text(4330, 5.5e-13, s=f"Fe I", rotation=90, fontsize=13, alpha=0.7)
ax_big_0.text(4360, 5.5e-13, s=f"Fe I", rotation=90, fontsize=13, alpha=0.7)
ax_big_0.text(4390, 5.5e-13, s=f"Fe I", rotation=90, fontsize=13, alpha=0.7)

for idx, row in iron.iterrows():
    x = row['WAVE']
    x = air_to_vac(row['WAVE']* u.AA)
    x = x.to_value(u.AA)
    if x < 5150 or x > 5720:
        continue
    ax_big_1.axvline(x, color='k', linestyle=':', alpha=0.2)
ax_big_1.text(5175, 5.8e-13, f"Fe I", rotation=90, fontsize=13, alpha=0.7)
ax_big_1.text(5275, 5.8e-13, f"Fe I", rotation=90, fontsize=13, alpha=0.7)
ax_big_1.text(5332, 5.8e-13, f"Fe I", rotation=90, fontsize=13, alpha=0.7)
ax_big_1.text(5379, 5.8e-13, f"Fe I", rotation=90, fontsize=13, alpha=0.7)
ax_big_1.text(5413, 5.8e-13, f"Fe I", rotation=90, fontsize=13, alpha=0.7)
ax_big_1.text(5552, 5.8e-13, f"Fe I", rotation=90, fontsize=13, alpha=0.7)
ax_big_1.text(5646, 5.8e-13, f"Fe I", rotation=90, fontsize=13, alpha=0.7)
ax_big_1.text(5683, 5.8e-13, f"Fe I", rotation=90, fontsize=13, alpha=0.7)
ax_big_1.text(5579, 5.8e-13, f"[OI]", rotation=90, fontsize=13, alpha=0.7)
ax_big_1.axvline(5578.3, 5.8e-13, color='k', linestyle='-', alpha=0.2)
ax_big_1.axvspan(5155.5, 5166.5, color='gray', alpha=0.1)
ax_big_1.text(5145, 5.8e-13, s=f"C$_2$", rotation=90, fontsize=13, alpha=0.7)

for idx, row in nickel.iterrows():
    x = row['WAVE']
    x = air_to_vac(row['WAVE'] * u.AA)
    x = x.to_value(u.AA)
    if x < 5150 or x > 5720:
        continue
    ax_big_1.axvline(x, color='k', linestyle='-.', alpha=0.2)
    ax_big_1.text(x+5, 5.8e-13, f"{row['name']} {row['ion']}", rotation=90, fontsize=13, alpha=0.7)

for idx, row in iron.iterrows():
    x = row['WAVE']
    x = air_to_vac(row['WAVE']* u.AA)
    x = x.to_value(u.AA)
    if x < 3801 or x > 3950:
        continue
    ax_empty.axvline(x, color='k', linestyle=':', alpha=0.2)

for idx, row in nickel.iterrows():
    x = row['WAVE']
    x = air_to_vac(row['WAVE']* u.AA)
    x = x.to_value(u.AA)
    if x < 3801 or x > 3950:
        continue
    ax_empty.axvline(x, color='k', linestyle='-.', alpha=0.2)
ax_empty.axvspan(3867, 3882.7, color='gray', alpha=0.1)
ax_empty.text(3869, 2.2e-12, s=f"CN", rotation=90, fontsize=13, alpha=0.7)
# plt.tight_layout() 

from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# --- Define custom legend elements ---
legend_elements = [
    Line2D([0], [0], color='crimson', lw=2, alpha=0.7,
           label='Post\nPerihelion'),
    Line2D([0], [0], color='steelblue', lw=2, alpha=0.7,
           label='Pre\nPerihelion'),
    Line2D([0], [0], color='gray', lw=2, linestyle=':', alpha=0.2,
           label='Fe I'),
    Line2D([0], [0], color='gray', lw=2, linestyle='-.', alpha=0.2,
           label='Ni I'),
    Line2D([0], [0], color='gray', lw=2, linestyle='-', alpha=0.2,
           label='[OI]'),
    Line2D([0], [0], color='gray', lw=2, linestyle='--', alpha=0.2,
           label='Molecules'),
    Patch(facecolor='gray', alpha=0.1, label='Molecules')
]
ax_empty.legend(handles=legend_elements, loc='upper right', fontsize=10)
fig.tight_layout()
fig.subplots_adjust(left=0.05, right=0.97, bottom=0.05, top=0.98)

plt.savefig('Plots/Super_plot_post_pre_peri.pdf', dpi=300)

plt.show()



/var/folders/qk/w_n1xnwj22176_k_hx_3sdf80000gn/T/ipykernel_95216/1405688913.py:4: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  iron = pd.read_csv('Element_lines/iron_lines_air.txt', delim_whitespace=True, names=['WAVE', 'name', 'ion'])
/var/folders/qk/w_n1xnwj22176_k_hx_3sdf80000gn/T/ipykernel_95216/1405688913.py:5: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  nickel = pd.read_csv('Element_lines/nickel_lines_air.txt', delim_whitespace=True, names=['WAVE', 'name', 'ion'])
/var/folders/qk/w_n1xnwj22176_k_hx_3sdf80000gn/T/ipykernel_95216/1405688913.py:577: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
